## 스케일링
- 컬럼 간 숫자의 상대적 크기 차이 때문에, 모델에 영향을 미치는 문제들이 달라진다. 이를 해결하기 위한 과정이다.

- 정규화, 표준화 이 두가지를 가장 많이 활용한다
- 사이킷런 라이브러리에 구현되어 있다

### 라이브러리 불러오기

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler  # 정규화, 표준화

### 데이터 불러오기

In [2]:
CSV_PATH = 'dataFile/Clean_Dataset.csv'
df_origin = pd.read_csv(CSV_PATH)

df_origin.head()

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


In [3]:
df_origin = df_origin.drop('Unnamed: 0', axis=1)    # 컬럼 삭제
df_origin.head()

,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


In [4]:
df = df_origin.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300153 entries, 0 to 300152
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   airline           300153 non-null  str    
 1   flight            300153 non-null  str    
 2   source_city       300153 non-null  str    
 3   departure_time    300153 non-null  str    
 4   stops             300153 non-null  str    
 5   arrival_time      300153 non-null  str    
 6   destination_city  300153 non-null  str    
 7   class             300153 non-null  str    
 8   duration          300153 non-null  float64
 9   days_left         300153 non-null  int64  
 10  price             300153 non-null  int64  
dtypes: float64(1), int64(2), str(8)
memory usage: 25.2 MB


In [5]:
# 수치형 데이터만 분리하여 데이터프레임 만들기
df_num = df[['duration', 'days_left', 'price']]
df_num.head()

,duration,days_left,price
0,2.17,1,5953
1,2.33,1,5953
2,2.17,1,5956
3,2.25,1,5955
4,2.33,1,5955


### 정규화 수식 적용하기 - 수식만으로 해보기

In [6]:
df_num_norm = (df_num - df_num.min()) / (df_num.max() - df_num.min())

In [7]:
df_num_norm.head()

,duration,days_left,price
0,0.027347,0.0,0.039749
1,0.030612,0.0,0.039749
2,0.027347,0.0,0.039773
3,0.028980,0.0,0.039765
4,0.030612,0.0,0.039765


### 사이킷런으로 동일 결과 재현하기 - MinMaxScaler

In [8]:
minmax = MinMaxScaler()
minmax.fit_transform(df_num)

array([[0.02734694, 0.        , 0.03974878],
       [0.03061224, 0.        , 0.03974878],
       [0.02734694, 0.        , 0.03977338],
       ...,
       [0.26530612, 1.        , 0.6394733 ],
       [0.18714286, 1.        , 0.65985603],
       [0.18877551, 1.        , 0.65985603]], shape=(300153, 3))

In [9]:
result1 = minmax.fit_transform(df_num)

# 데이터프레임으로 만들어서 결과 확인
df_num_norm_sklearn = pd.DataFrame(
    result1,
    columns=df_num.columns
)
df_num_norm_sklearn.head()

,duration,days_left,price
0,0.027347,0.0,0.039749
1,0.030612,0.0,0.039749
2,0.027347,0.0,0.039773
3,0.028980,0.0,0.039765
4,0.030612,0.0,0.039765


### 표준화 수식 적용하기
- 정확히 0이 아니라 아주 작은 값으로 나오는 이유
    - 부동소수점 연산 오차 문제
    - 정규화와 다르게 0/1로 고정되어 있지 않고, 데이터 분포에 따라 달라진다는 특징도 있다

- 이상치가 많거나 데이터 범위가 정해지지 않았다면 표준화가 더 안전한 편
- 데이터 범위가 명확히 정해져있고, 이상치가 없다면 정규화도 무방하다

In [10]:
df_num_std = (df_num - df_num.mean()) / df_num.std()
df_num_std.head()

,duration,days_left,price
0,-1.397528,-1.843872,-0.658067
1,-1.375282,-1.843872,-0.658067
2,-1.397528,-1.843872,-0.657935
3,-1.386405,-1.843872,-0.657979
4,-1.375282,-1.843872,-0.657979


### 사이킷런으로 표준화해보기 - StandardScaler

In [11]:
std_scaler = StandardScaler()
result2 = std_scaler.fit_transform(df_num)
result2

array([[-1.39753079, -1.84387477, -0.65806849],
       [-1.3752838 , -1.84387477, -0.65806849],
       [-1.39753079, -1.84387477, -0.65793631],
       ...,
       [ 0.22371837,  1.69569214,  2.56454459],
       [-0.30881888,  1.69569214,  2.67407096],
       [-0.29769538,  1.69569214,  2.67407096]], shape=(300153, 3))

In [13]:
# 데이터프레임으로 결과 확인
df_num_std_sklearn = pd.DataFrame(
    result2,
    columns=df_num.columns
)
df_num_std_sklearn.head()

,duration,days_left,price
0,-1.397531,-1.843875,-0.658068
1,-1.375284,-1.843875,-0.658068
2,-1.397531,-1.843875,-0.657936
3,-1.386407,-1.843875,-0.657980
4,-1.375284,-1.843875,-0.657980
